# ACCeSS 风格的 T / D / L 三模式分析（MPQUIC 日志）

对照 IWQoS 2022 **ACCeSS** 论文的思路，在**同一网络场景**下比较三种 **utility 模式**（吞吐 **T**、时延 **D**、丢包 **L**），而不是 Phase2 里「baseline / delay / loss 三种网络包」。

## 你需要准备的数据

三次独立实验，**Mininet / `tc` 脚本与码率尽量一致**，仅改变客户端 **utility mode**（例如环境变量或构建参数）：

- `vm_run_…_T/` — `pull_*.log` 内 `[meta] utility_mode=T` 且 `[utility] mode=T`
- `vm_run_…_D/` — 同上，`D`
- `vm_run_…_L/` — 同上，`L`

每个目录可含 `tc_delay_*.log` / `tc_loss_*.log`（与 Phase2 相同格式）。**TC 竖线**由 `load_labeled_vm_runs` 从 `tc_from_label` 指定目录读取；若 **T 目录没有 tc 文件**（例如 baseline），notebook 会自动在 **D/L 目录**里找 `tc_delay` / `tc_loss` 与对应 `pull` 做时间对齐。

> **注意**：下方默认路径把 Phase2 的 baseline / delay / loss **三种不同网络** 标成 T/D/L，只为**演示脚本**；论文级对比应使用 **同一 tc 场景** 下三次运行、仅 **utility_mode** 不同。
>
> 客户端已支持：`-utility-mode=T|D|L`（见 `main.go`）。三次实验除该参数外保持一致，例如：  
> `./4dmap -utility-mode=D -type=true ...`（拉流）与推流端同样带 `-utility-mode=D`。

## 本 notebook 产出

| 输出 | 含义（与 ACCeSS 叙事对齐） |
|------|---------------------------|
| 四联图 PDF | ① 聚合带宽 Σpath ② 平均 OWD ③ 平均 U ④ 各时刻 max(loss) — **三条线 = T/D/L** |
| G/D/L 分解图 | 归一化分量随时间变化 |
| 汇总表 | 全程 mean 指标，便于写「Mode T 比 D 高 x%」类句子 |

**横轴**：各 pull 日志 **第一条 `[utility]`** 起的相对秒（三次 run 的 t=0 不同，对比的是各自「开跑后」的形态，与论文中单次 300s 曲线可读性一致；若要严格对齐墙钟需后处理）。

等价 CLI（无需 notebook）：  
`python scripts/analyze/plot_phase2.py --tdl <T_dir> <D_dir> <L_dir>`


In [5]:
from pathlib import Path
import sys

REPO = Path("../..").resolve()
sys.path.insert(0, str(Path(".").resolve()))

import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

from parse_logs import load_labeled_vm_runs, estimate_tc_pull_offset_seconds
from plot_phase2 import (
    plot_tdl_access_style,
    plot_tdl_utility_components,
    summarize_tdl_modes,
    COLORS_TDL,
)

FIGURES = REPO / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("REPO:", REPO)

REPO: /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment


## 1. 配置三个 `vm_run_*` 目录

改成你机器上 **T、D、L** 三次实验的路径。若暂时没有三次数据，可先用**同一目录占位**验证管线（曲线会重合）。

`tc_from_label`：从哪个模式的文件夹读 `tc_*.log`（默认用 `T`）。

In [6]:
LOG_UNDER = REPO / "logs_exp" / "log"  # 与 phase2 一致

RUN_T = LOG_UNDER / "vm_run_20260402_165427"
RUN_D = LOG_UNDER / "vm_run_20260402_165638"
RUN_L = LOG_UNDER / "vm_run_20260402_165833"

label_to_dir = {"T": RUN_T, "D": RUN_D, "L": RUN_L}
tc_from_label = "T"

for k, p in label_to_dir.items():
    pull = next(Path(p).glob("pull_*.log"), None)
    print(f"  [{k}] {p}  pull={'OK' if pull else 'MISSING'}")

  [T] /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment/logs_exp/log/vm_run_20260402_165427  pull=OK
  [D] /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment/logs_exp/log/vm_run_20260402_165638  pull=OK
  [L] /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment/logs_exp/log/vm_run_20260402_165833  pull=OK


In [7]:
import re as _re_meta

_re_meta_line = _re_meta.compile(r"utility_mode=([TDL])")
_re_util_mode = _re_meta.compile(r"\[utility\] path=\d+ mode=([TDL])")


def _first_utility_mode_from_file(path: Path, max_bytes: int = 8_000_000):
    """Scan pull log for first [utility] mode= (meta 只在开头，可能读不到)."""
    buf = []
    n = 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            buf.append(line)
            n += len(line.encode("utf-8", errors="replace"))
            if n > max_bytes:
                break
    text = "".join(buf)
    m = _re_util_mode.search(text)
    return m.group(1) if m else None


for k, p in label_to_dir.items():
    pull = next(Path(p).glob("pull_*.log"), None)
    if not pull:
        print(f"[{k}] no pull log")
        continue
    with open(pull, encoding="utf-8", errors="replace") as f:
        head = "".join(f.readline() for _ in range(200))
    mm = _re_meta_line.search(head)
    meta_m = mm.group(1) if mm else None
    util_m = _first_utility_mode_from_file(pull)
    ok = k == meta_m == util_m
    flag = "OK" if ok else "⚠ 不一致"
    print(f"[{k}] {flag}  [meta] utility_mode={meta_m or '?'}  [utility] mode={util_m or '?'}")
    if not ok:
        print(
            "     → 说明：日志里实际跑的仍是 " + str(util_m or meta_m or "?")
            + "；文件夹标签 " + k + " 只是 notebook 里的名字。"
        )
        print(
            "     → 要做真·T/D/L 对比：请在 VM 用 D/L 启动客户端（如环境变量/配置设 utility_mode=D/L），"
            "再各存一份 vm_run_*。"
        )

_bad = False
for lab in label_to_dir:
    _pl = next(Path(label_to_dir[lab]).glob("pull_*.log"), None)
    _um = _first_utility_mode_from_file(_pl) if _pl else None
    if _um != lab:
        _bad = True
        break
if _bad:
    print(
        "\n【结论】当前三套日志并非三种 utility 模式；"
        "曲线对比的是「不同网络 Phase2 包」或重复 T，解读时不要写成 ACCeSS 式 T/D/L。"
    )

[T] OK  [meta] utility_mode=T  [utility] mode=T
[D] ⚠ 不一致  [meta] utility_mode=T  [utility] mode=T
     → 说明：日志里实际跑的仍是 T；文件夹标签 D 只是 notebook 里的名字。
     → 要做真·T/D/L 对比：请在 VM 用 D/L 启动客户端（如环境变量/配置设 utility_mode=D/L），再各存一份 vm_run_*。
[L] ⚠ 不一致  [meta] utility_mode=T  [utility] mode=T
     → 说明：日志里实际跑的仍是 T；文件夹标签 L 只是 notebook 里的名字。
     → 要做真·T/D/L 对比：请在 VM 用 D/L 启动客户端（如环境变量/配置设 utility_mode=D/L），再各存一份 vm_run_*。

【结论】当前三套日志并非三种 utility 模式；曲线对比的是「不同网络 Phase2 包」或重复 T，解读时不要写成 ACCeSS 式 T/D/L。


## 2. 加载与 TC 时间对齐

In [8]:
df_util, df_mon, tc_steps = load_labeled_vm_runs(label_to_dir, tc_from_label=tc_from_label)

print("utility rows:", len(df_util), " monitor rows:", len(df_mon))
print("labels:", sorted(df_util["label"].unique()))
print("tc_steps keys:", list(tc_steps.keys()))

tc_offsets = {"delay": 0.0, "loss": 0.0}

def _first_pull_tc(glob_tc: str):
    for _d in sorted(label_to_dir.values(), key=lambda x: str(x)):
        pls, tcs = list(Path(_d).glob("pull_*.log")), list(Path(_d).glob(glob_tc))
        if pls and tcs:
            return pls[0], tcs[0]
    return None, None

if "delay" in tc_steps and not tc_steps["delay"].empty:
    pull_src, tcd = _first_pull_tc("tc_delay_*.log")
    if pull_src:
        tc_offsets["delay"] = float(estimate_tc_pull_offset_seconds(pull_src, tcd))
if "loss" in tc_steps and not tc_steps["loss"].empty:
    pull_src, tcl = _first_pull_tc("tc_loss_*.log")
    if pull_src:
        tc_offsets["loss"] = float(estimate_tc_pull_offset_seconds(pull_src, tcl))
print("TC_OFFSET_DICT:", tc_offsets)

utility rows: 588548  monitor rows: 588551
labels: ['D', 'L', 'T']
tc_steps keys: []
TC_OFFSET_DICT: {'delay': 0.0, 'loss': 0.0}


## 3. 汇总表（全程均值）

写论文时：Mode **T** 应 **mean_bw** 更高；**D** 应 **mean_owd** 更低（在相同扰动下）；**L** 应 **mean_loss** 更低 — 实际结果取决于实现与场景，以表为准。

In [9]:
summary = summarize_tdl_modes(df_util)
print(summary.to_string(index=False))
summary.to_csv(FIGURES / "access_tdl_summary.csv", index=False)
print("saved", FIGURES / "access_tdl_summary.csv")

mode  n_rows  mean_bw_mbps  mean_owd_ms  mean_loss   mean_U  mean_gain  mean_backoff
   D  190118     20.867784    77.869623        0.0 0.103407   1.036327      0.986553
   L  205641     27.444261    69.964763        0.0 0.145087   1.050077      0.981070
   T  192789     19.463283    89.233511        0.0 0.091774   1.032617      0.988400
saved /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment/figures/access_tdl_summary.csv


## 4. 四联图（ACCeSS 式：吞吐 / 时延 / 效用 / 丢包）

In [10]:
plot_tdl_access_style(
    df_util,
    tc_steps,
    FIGURES / "access_tdl_mpquic.pdf",
    tc_offsets=tc_offsets,
    title="MPQUIC utility modes: T (throughput) vs D (delay) vs L (loss)",
)
plot_tdl_utility_components(
    df_util,
    tc_steps,
    FIGURES / "access_tdl_GDL.pdf",
    tc_offsets=tc_offsets,
)

[plot_tdl] saved → /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment/figures/access_tdl_mpquic.pdf
[plot_tdl] saved → /Users/jememalum/Project/mpquic/qcurl-4dmap-experiment/figures/access_tdl_GDL.pdf


## 5. （可选）事件窗统计 — 仿论文「100–110s 提升 x%」

下面示例：在 `t ∈ [t0, t1]` 内对**聚合带宽**求均值，再与参考窗比较。按你的 `tc_*.log` 修改 `t0, t1, t_ref0, t_ref1`。

In [11]:
from plot_phase2 import _aggregate_metric_per_label


def window_mean_bw(lbl: str, t_lo: float, t_hi: float) -> float:
    s = _aggregate_metric_per_label(df_util, lbl, "bw_mbps", "sum")
    seg = s[(s.index >= t_lo) & (s.index <= t_hi)]
    return float(seg.mean()) if len(seg) else float("nan")


# 示例：第二个 delay 台阶后 10s（按你实验改数字）
t0, t1 = 25.0, 35.0
t_ref0, t_ref1 = 5.0, 15.0

rows = []
for lbl in sorted(df_util["label"].unique()):
    a = window_mean_bw(lbl, t0, t1)
    b = window_mean_bw(lbl, t_ref0, t_ref1)
    pct = (a - b) / b * 100.0 if b and b == b else float("nan")
    rows.append({"mode": lbl, f"mean_bw_[{t0},{t1}]": a, f"mean_bw_[{t_ref0},{t_ref1}]": b, "pct_change": pct})
pd.DataFrame(rows)

,mode,"mean_bw_[25.0,35.0]","mean_bw_[5.0,15.0]",pct_change
0,D,64.205976,62.323287,3.020844
1,L,82.904348,100.864959,-17.806591
2,T,66.686712,66.287178,0.602732


## 图例颜色

- **T**：红 — 吞吐优先  
- **D**：蓝 — 时延优先  
- **L**：绿 — 丢包优先  

与 ACCeSS-T 论文用色接近，便于答辩时口头对应。